In [2]:
import pandas as pd          # manipulação e análise de dados
import numpy as np           # funções matemáticas e arrays (usado para NaN)

# =============================================================================
# CONFIGURAÇÃO DO REPOSITÓRIO GITHUB (CAMINHO RAW)
# =============================================================================
USER = "lpvianna81"                         # nome do usuário no GitHub
REPO = "projetofinal_tecnicas_python"       # nome do repositório
BRANCH = "main"                             # branch utilizada
# Monta a URL base para acessar arquivos brutos (raw) no GitHub
BASE_URL = f"https://raw.githubusercontent.com/{USER}/{REPO}/{BRANCH}/"


def executar_projeto_f1():
    """
    Função principal que executa todo o pipeline:
    - Carga dos CSVs
    - Renomeação de colunas
    - Merge (junção) das tabelas
    - Tratamento de qualidade (limpeza e conversão de tipos)
    - Geração das 10 análises solicitadas
    """
    print(f"Conectando ao repositório: {REPO}...\n")

    try:
        # =========================================================================
        # 1. CARGA DOS DADOS COM RENOMEAÇÃO DE COLUNAS
        # =========================================================================
        # results.csv: contém resultados de cada piloto em cada corrida
        results = pd.read_csv(f"{BASE_URL}results.csv")

        # drivers.csv: dados dos pilotos (nome, nacionalidade, data de nascimento)
        # Renomeia a coluna 'nationality' para 'driver_nationality' para evitar
        # conflito com a coluna de nacionalidade dos construtores (team_nationality)
        drivers = pd.read_csv(f"{BASE_URL}drivers.csv").rename(columns={
            'nationality': 'driver_nationality'
        })
        # Cria uma coluna com o nome completo do piloto (primeiro nome + sobrenome)
        drivers['driver_full_name'] = drivers['forename'] + ' ' + drivers['surname']

        # races.csv: informações das corridas (data, nome do circuito, ano)
        # Renomeia 'name' para 'race_name' para ficar mais claro
        races = pd.read_csv(f"{BASE_URL}races.csv").rename(columns={
            'name': 'race_name'
        })

        # constructors.csv: dados das equipes/construtores
        # Renomeia 'name' para 'team_name' e 'nationality' para 'team_nationality'
        constructors = pd.read_csv(f"{BASE_URL}constructors.csv").rename(columns={
            'name': 'team_name',
            'nationality': 'team_nationality'
        })

        # status.csv: descrição dos status de corrida (Finished, Accident, etc.)
        status = pd.read_csv(f"{BASE_URL}status.csv")

        # =========================================================================
        # 2. MERGE DOS DATASETS (JUNÇÃO DAS TABELAS)
        # =========================================================================
        # Realiza junções sequenciais (INNER JOIN) usando as chaves apropriadas.
        # O resultado final 'df' contém uma tabela desnormalizada com todas as
        # informações relevantes para cada registro de resultado de corrida.
        df = results.merge(drivers, on='driverId') \
                    .merge(races, on='raceId') \
                    .merge(constructors, on='constructorId') \
                    .merge(status, on='statusId')

        # =========================================================================
        # 3. TRATAMENTO DE QUALIDADE (LIMPEZA E CONVERSÃO DE TIPOS)
        # =========================================================================
        # Substitui a string '\N' (usada nos CSVs para representar nulo) por NaN
        df.replace(r'\N', np.nan, inplace=True)

        # Converte a coluna 'points' para numérico; valores inválidos viram NaN,
        # e então preenchemos esses NaN com 0 (piloto sem pontos na corrida)
        df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)

        # Converte 'grid' (posição de largada) para numérico
        df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
        # Converte 'positionOrder' (posição final de chegada) para numérico
        df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')

        # Converte as colunas de data para o tipo datetime (data/hora)
        # 'date' = data da corrida
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        # 'dob' = data de nascimento do piloto
        df['dob'] = pd.to_datetime(df['dob'], errors='coerce')

        print("Dados carregados e limpos com sucesso.\n")

        # =========================================================================
        # 4. ANÁLISES (AS 10 PERGUNTAS)
        # =========================================================================
        print("=" * 50)
        print("RELATÓRIO DE ANÁLISE TÉCNICA - F1")
        print("=" * 50)

        # Q1: Os 10 pilotos com mais vitórias na história
        # Filtra apenas as linhas onde positionOrder == 1 (vitória),
        # conta quantas vezes cada piloto aparece e mostra os 10 primeiros.
        print("\n1. Os 10 pilotos com mais vitórias na história:")
        print(df[df['positionOrder'] == 1]['driver_full_name'].value_counts().head(10))

        # Q2: As 5 equipes com mais pontos acumulados (todos os anos)
        # Agrupa por nome da equipe, soma os pontos, ordena decrescente e pega top5.
        print("\n2. As 5 equipes com mais pontos acumulados:")
        print(df.groupby('team_name')['points'].sum().sort_values(ascending=False).head(5))

        # Q3: Os 10 pilotos com mais Pole Positions
        # Pole position = grid == 1 (largada em primeiro lugar).
        print("\n3. Os 10 pilotos com mais Pole Positions:")
        print(df[df['grid'] == 1]['driver_full_name'].value_counts().head(10))

        # Q4: Os 10 circuitos que mais sediaram GPs
        # Agrupa por nome da corrida (circuito), conta o número de IDs de corrida
        # distintos (nunique) e ordena decrescente.
        print("\n4. Os 10 circuitos que mais sediaram GPs:")
        print(df.groupby('race_name')['raceId'].nunique().sort_values(ascending=False).head(10))

        # Q5: Média de idade dos campeões por década
        # Primeiro, agrupa por ano, piloto e data de nascimento, somando os pontos
        campeoes_db = df.groupby(['year', 'driverId', 'dob'])['points'].sum().reset_index()
        # Para cada ano, encontra o índice do piloto com maior pontuação (campeão)
        idx_campeoes = campeoes_db.groupby(['year'])['points'].idxmax()
        # Seleciona essas linhas -> campeões de cada ano
        campeoes_finais = campeoes_db.loc[idx_campeoes].copy()
        # Calcula a idade do campeão no ano do título (ano - ano de nascimento)
        campeoes_finais['age'] = campeoes_finais['year'] - campeoes_finais['dob'].dt.year
        # Cria a coluna 'decade' (ex: 1990 para os anos 1990-1999)
        campeoes_finais['decade'] = (campeoes_finais['year'] // 10) * 10
        # Agrupa por década e calcula a média das idades
        print("\n5. Média de idade dos campeões por década:")
        print(campeoes_finais.groupby('decade')['age'].mean())

        # Q6: Piloto com mais Voltas Mais Rápidas (Rank 1)
        # 'rank' igual a '1' normalmente indica melhor volta da corrida.
        print("\n6. Piloto com mais Voltas Mais Rápidas (Rank 1):")
        print(df[df['rank'] == '1']['driver_full_name'].value_counts().head(1))

        # Q7: Número de vitórias em corridas por nacionalidade do piloto
        # Filtra vitórias (positionOrder == 1) e conta as nacionalidades.
        print("\n7. Número de vitórias em corridas por nacionalidade do piloto:")
        print(df[df['positionOrder'] == 1]['driver_nationality'].value_counts())

        # Q8: Evolução de pontos da Ferrari (Últimas 20 temporadas)
        ultimo_ano = df['year'].max()                     # ano mais recente nos dados
        # Filtra apenas a equipe Ferrari e anos posteriores a (último ano - 20)
        ferrari = df[(df['team_name'] == 'Ferrari') & (df['year'] > (ultimo_ano - 20))]
        print("\n8. Evolução de pontos da Ferrari (Últimas 20 temporadas):")
        print(ferrari.groupby('year')['points'].sum())

        # Q9: Top 3 Pilotos Brasileiros com mais vitórias
        # Filtra nacionalidade 'Brazilian' e vitórias, depois conta por nome.
        br_vits = df[(df['driver_nationality'] == 'Brazilian') & (df['positionOrder'] == 1)]
        print("\n9. Top 3 Pilotos Brasileiros com mais vitórias:")
        print(br_vits['driver_full_name'].value_counts().head(3))

        # Q10: Top 10 pilotos com mais GPs disputados
        # value_counts() em 'driver_full_name' conta quantas vezes cada piloto aparece
        # (cada linha é uma participação em corrida).
        print("\n10. Top 10 pilotos com mais GPs disputados:")
        print(df['driver_full_name'].value_counts().head(10))

    except Exception as e:
        # Captura qualquer erro que ocorra durante o pipeline e exibe a mensagem
        print(f"Erro na execução do pipeline: {e}")


# =============================================================================
# PONTO DE ENTRADA DO SCRIPT
# =============================================================================
# Se este arquivo for executado diretamente (não importado como módulo),
# chama a função principal.
if __name__ == "__main__":
    executar_projeto_f1()

Conectando ao repositório: projetofinal_tecnicas_python...

Dados carregados e limpos com sucesso.

RELATÓRIO DE ANÁLISE TÉCNICA - F1

1. Os 10 pilotos com mais vitórias na história:
driver_full_name
Lewis Hamilton        105
Michael Schumacher     91
Max Verstappen         63
Sebastian Vettel       53
Alain Prost            51
Ayrton Senna           41
Fernando Alonso        32
Nigel Mansell          31
Jackie Stewart         27
Niki Lauda             25
Name: count, dtype: int64

2. As 5 equipes com mais pontos acumulados:
team_name
Ferrari     11091.27
Mercedes     7730.64
Red Bull     7673.00
McLaren      7022.50
Williams     3641.00
Name: points, dtype: float64

3. Os 10 pilotos com mais Pole Positions:
driver_full_name
Lewis Hamilton        104
Michael Schumacher     68
Ayrton Senna           65
Sebastian Vettel       57
Max Verstappen         40
Jim Clark              34
Alain Prost            33
Nigel Mansell          32
Nico Rosberg           30
Juan Fangio            29
Name: